# Transfering to the New Environment

Due to big success of previous mining, Ms. Dixon has been given task to open new mine. Ms. Dixon can easily use the same coordinate system but she can expect that the output produced in the new area will not be exactly the same.  Ms. Dixon does not know it, but the gold production in the candidate area, where she is assigned to set up new mine, is linearly related to the area that she already explored. 


## New Plot of Land and Its Secret Map

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ProcessOptimizer
from ProcessOptimizer.model_systems import get_model_system

gold_map = get_model_system('gold_map')
coordinates = [(0.0, 15.0), (0.0, 15.0)]
bounds = gold_map.space.bounds

# experiment with setting the rotate and shift parameters to get new gold map candidates
# rotate = 1. and shift = 0. corresponds to the original gold map
new_gold_map = get_model_system('candidate_gold_map', rotate=3, shift=2.5)
candidate_bounds = gold_map.space.bounds

In [ ]:
x_list = np.linspace(coordinates[0][0],coordinates[0][1],100)
y_list = np.linspace(coordinates[1][0],coordinates[1][1],15,100)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

(x_mesh,y_mesh) = np.meshgrid(x_list,y_list)
mesh_values = lambda score_fcn: [[-score_fcn.get_score((x,y)) for (x,y) in zip(x_vec,y_vec)] 
                for (x_vec,y_vec) in zip(x_mesh,y_mesh)]
    
for ax_, score_fcn_, title_ in zip(ax.flatten(), 
                                       [gold_map, new_gold_map], 
                                       ['Secret Gold Map', 'Secret New Gold Opportunity Map']):
    ax_.contourf(x_mesh,y_mesh, mesh_values(score_fcn_), 10)
    ax_.set_xlabel("Distance east-west in km")
    ax_.set_ylabel("Distance north-south in km")
    ax_.set_title(title_)

## Setting Up New Environment

Recall the process of exploring Gold Map Environment: 
1. Start to Dig 
2. More Digging 
3. Collect  

As this is first attempt to explore the space, Ms. Dixon send there unexperienced team to perform some preliminary digging. 



In [ ]:
first_dig_location = [12.5, 7.5]
first_gold_found = gold_map.get_score(first_dig_location)
first_gold_found_new_plot = new_gold_map.get_score(first_dig_location)
 
print("Location: %s"%(first_dig_location))
print("Amount of gold found in the old plot: %s"%(-first_gold_found))
print("Amount of gold found in the new plot: %s"%(-first_gold_found_new_plot))

This random dig was very successful compared to the results from the previous environmemnt. The team of newbies enthusiastically continued to dig. 

In [ ]:
import ProcessOptimizer as op
from ProcessOptimizer.learning import GaussianProcessRegressor
from ProcessOptimizer.learning.gaussian_process import kernels 
# 
opt_new_plot = ProcessOptimizer.Optimizer(
            dimensions=coordinates, 
            # we like to have more control over the parameters of the Gaussian Process
            base_estimator=GaussianProcessRegressor(
                kernel=kernels.ConstantKernel(constant_value=1.5)*kernels.RBF(length_scale=0.5, length_scale_bounds=(1e-2, 1e2)),
                #kernel=kernels.ConstantKernel()*kernels.RBF(length_scale_bounds=(1e-2, 1e2)),
                normalize_y=True,
                noise="gaussian",
                n_restarts_optimizer=50, 
                # The number of restarts of the optimizer for finding the kernel’s parameters which maximize the log-marginal likelihood. 
                # The first run of the optimizer is performed from the kernel’s initial parameters, the remaining ones (if any) from thetas sampled log-uniform randomly from the space of allowed theta-values. 
                # If greater than 0, all bounds must be finite. 
                # n_restarts_optimizer == 0 implies that one run is performed
        ), 
            acq_optimizer="lbfgs")

# opt_new_plot = ProcessOptimizer.Optimizer(coordinates, 
#             # we like to have more control over the parameters of the Gaussian Process
#             "GP", n_restarts_optimizer = 50, acq_optimizer="lbfgs", n_initial_points=15)

get_gp_params = lambda opt_: opt_.base_estimator_.get_params()

get_gp_params(opt_new_plot)

In [ ]:
# Finding the first position
new_coordinates = opt_new_plot.ask()
print(f"ProcessOptimizer suggests we start digging at {new_coordinates}")
# Digging at the suggested coordinates
gold_found = new_gold_map.get_score(new_coordinates)
# Telling the ProcessOptimizer about how much gold we found
opt_new_plot.tell(new_coordinates, gold_found)
print(f"ProcessOptimizer knows about a dig at {opt_new_plot.Xi[0]} where {-opt_new_plot.yi[0]} mg of gold was found.")

In [ ]:
# For each of the next 9 positions
for index in range(20):
    # Find the place to dig
    new_dig_site = opt_new_plot.ask()
    # Digging for gold
    gold_found = new_gold_map.get_score(new_dig_site)
    # Telling the optimiser how uch gold we found
    result_new_plot = opt_new_plot.tell(new_dig_site, gold_found)
    print(f"We dug at {new_dig_site} and found {-gold_found} mg of gold")
# Telling the optimiser about the very first dig
opt_new_plot.tell([7.5,7.5],first_gold_found)
# plotting
ProcessOptimizer.plot_objective(result=result_new_plot,pars="expected_minimum",dimensions = ["E-W","N-S"]);

In [ ]:
get_gp_params(opt_new_plot)

In [ ]:
for i in range(20):
    # Finding new dig site.
    new_dig_site=opt_new_plot.ask()
    # Digging and finding gold.
    gold_found = new_gold_map.get_score(new_dig_site)
    # Telling ProcessOptimizer how much gold we found at the new dig site
    opt_new_plot.tell(new_dig_site,gold_found)
# plotting
ProcessOptimizer.plot_objective(result=result_new_plot,pars="expected_minimum",dimensions = ["E-W","N-S"]);\
    

In [ ]:
opt_new_plot.models[-1].fit(opt_new_plot.Xi, opt_new_plot.yi)